# Multiperiod HPR Shared Design
This notebook shows the opt-in shared-design mode for HPR targeting on `crude_preheat_train_multiperiod.json`. The normal period-specific targeting calls still work, while `HPR_MULTIPERIOD_OPTIMIZATION_ENABLED` asks the HPR optimiser to choose one design vector across all canonical periods using the case weights.

**Support notice:** `PinchProblem` and `PinchWorkspace` are public package-root workflows. Concrete owner modules used by this advanced notebook remain unsupported; `from OpenPinch.main import pinch_analysis_service` is the strict integration contract.


## Case context
The crude preheat train multiperiod case has three named operating periods. A shared HPR design is useful when equipment is sized once, but operated against the `turndown`, `base`, and `peak` process states.


In [ ]:
import pandas as pd

from OpenPinch import PinchProblem
from OpenPinch.domain.enums import HPRcycle
from OpenPinch.domain._value.resolution import get_scalar_value

CASE_NAME = "crude_preheat_train_multiperiod.json"
PROJECT_NAME = "crude_hpr_multiperiod"
HPR_OPTIONS = {
    "HPR_TYPE": HPRcycle.CascadeCarnot.value,
    "HPR_LOAD_MODE": "fraction",
    "HPR_LOAD_FRACTION": 0.25,
    "HPR_MAX_MULTISTART": 1,
    "HPR_N_COND": 1,
    "HPR_N_EVAP": 1,
    "HPR_BB_MINIMISER": "rbf_surrogate",
}


## Inspect the period set
The weights below are used for operating cost, feasibility penalty, and `summary_frame(periods="weighted_average", ...)`. Shared simulated-cycle ranking adds the maximum annualized capital cost across periods so one design is sized for the peak-capital state.


In [ ]:
base_problem = PinchProblem(CASE_NAME, project_name=PROJECT_NAME)
base_problem.update_options(HPR_OPTIONS)

{
    "period_ids": base_problem.period_ids,
    "weights": dict(zip(base_problem.period_ids, base_problem.master_zone.weights)),
}


## Period-specific HPR targeting
With the multi-period flag left off, each explicit `period_id` call solves the HPR placement for that one period. This is the established behaviour and remains the starting point for diagnostics.


In [ ]:
period_specific_rows = []

for period_id in base_problem.period_ids:
    target = base_problem.target.direct_heat_pump(period_id=period_id)
    period_specific_rows.append(
        {
            "period_id": period_id,
            "hpr_utility_total": get_scalar_value(target.hpr_utility_total),
            "hpr_work": get_scalar_value(target.hpr_work),
            "hpr_cop": get_scalar_value(target.hpr_cop),
            "hpr_success": target.hpr_success,
        }
    )

pd.DataFrame(period_specific_rows)


## Shared HPR design across all periods
Turning on `HPR_MULTIPERIOD_OPTIMIZATION_ENABLED` keeps the internal parent accessor shape the same, but the HPR optimiser evaluates each candidate design against every canonical period. The returned target row is still the requested selected period, and `hpr_details` carries the shared design context.


In [ ]:
shared_problem = PinchProblem(CASE_NAME, project_name=PROJECT_NAME)
shared_problem.update_options(
    HPR_OPTIONS | {"HPR_MULTIPERIOD_OPTIMIZATION_ENABLED": True}
)

shared_target = shared_problem.target.direct_heat_pump(period_id="base")
details = shared_target.hpr_details

{
    "selected_period": shared_target.period_id,
    "shared_period_ids": details.period_ids,
    "shared_period_weights": details.period_weights,
    "design_vector_length": len(details.design_vector),
    "has_weighted_output": details.weighted_output is not None,
}


In [ ]:
pd.DataFrame(
    [
        {
            "period_id": period_id,
            "weighted_objective": get_scalar_value(result.obj),
            "hpr_utility_total": get_scalar_value(result.utility_tot),
            "hpr_work": get_scalar_value(result.w_net),
            "hpr_cop": get_scalar_value(result.cop_h),
        }
        for period_id, result in details.period_outputs.items()
    ]
)


## Weighted HPR summaries
`summary_frame(periods="weighted_average", ...)` replays each period on an isolated baseline-zone copy. HPR operating fields are weighted, capital fields use their period maximum, and total annualized cost is recomputed from weighted operating cost plus maximum annualized capital. `all_with_weighted_average` keeps the canonical periods and appends the aggregate row.


In [ ]:
shared_problem.summary_frame(periods="weighted_average", detailed=True)[
    [
        "Target",
        "Period ID",
        "HPR Utility Total (value)",
        "HPR Work (value)",
        "HPR COP (value)",
    ]
].dropna(subset=["HPR Utility Total (value)"])


In [ ]:
shared_problem.summary_frame(periods="all_with_weighted_average", detailed=True)[
    [
        "Target",
        "Period ID",
        "HPR Utility Total (value)",
        "HPR Work (value)",
        "HPR COP (value)",
    ]
].dropna(subset=["HPR Utility Total (value)"])


## Interpretation
Use the period-specific table when you want to compare independent optima. Use the shared-design table when the equipment should be sized once across all operating states. The weighted summary row is the reporting view of that shared design, not an average of independently optimised HPR designs.
